In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Adjust paths to register local src engine
sys.path.append(os.path.abspath(os.path.join('..')))
from src.processing import calculate_noise_reduction

# Read historical baseline file directly from local storage disc
raw_data_path = "/home/soq/__shutupandbendover/devansh-swc-wala/market_data.parquet"
df_raw = pd.read_parquet(raw_data_path)
total_days = len(df_raw)
print(f"Setup Complete. Loaded {total_days} clean baseline historical points.")

ModuleNotFoundError: No module named 'plotly'

In [ ]:
# Prompt parameters
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Input parameters
start_day = int(input(f"Enter START day number (1 to {total_days}): "))
end_day = int(input(f"Enter END day number ({start_day} to {total_days}): "))
window_size = int(input("Enter smoothing rolling size array parameter: "))

# Calculate all 3 smoothing methods simultaneously
df_sma = calculate_noise_reduction(df_raw, method='SMA', window=window_size)
df_ema = calculate_noise_reduction(df_raw, method='EMA', window=window_size)
df_med = calculate_noise_reduction(df_raw, method='Median', window=window_size)

# Combine into a single analysis DataFrame
df_calc = df_raw.copy()
df_calc['Stock_SMA'] = df_sma['Stock_Smoothed']
df_calc['Stock_EMA'] = df_ema['Stock_Smoothed']
df_calc['Stock_Median'] = df_med['Stock_Smoothed']
df_calc['Day_Number'] = np.arange(1, len(df_calc) + 1)

# Apply dynamic range filter slice vectors
days_vector = np.arange(start_day, end_day + 1)
df_filtered = df_calc[df_calc['Day_Number'].isin(days_vector)]

# Generate Plotly Visualizations (All Methods Overlay)
fig = make_subplots(specs=[[{"secondary_y": True}]])

# 1. Asset Raw Price (Faint Background)
fig.add_trace(
    go.Scatter(
        x=df_filtered['Day_Number'], 
        y=df_filtered['Stock_Raw'], 
        name="Asset Raw Price", 
        line=dict(color="rgba(150, 150, 150, 0.4)", width=1.5, dash="dot")
    ), 
    secondary_y=False
)

# 2. SMA Line
fig.add_trace(
    go.Scatter(
        x=df_filtered['Day_Number'], 
        y=df_filtered['Stock_SMA'], 
        name=f"SMA ({window_size})", 
        line=dict(color="#1f77b4", width=2)
    ), 
    secondary_y=False
)

# 3. EMA Line
fig.add_trace(
    go.Scatter(
        x=df_filtered['Day_Number'], 
        y=df_filtered['Stock_EMA'], 
        name=f"EMA ({window_size})", 
        line=dict(color="#ff7f0e", width=2)
    ), 
    secondary_y=False
)

# 4. Median Line
fig.add_trace(
    go.Scatter(
        x=df_filtered['Day_Number'], 
        y=df_filtered['Stock_Median'], 
        name=f"Median ({window_size})", 
        line=dict(color="#2ca02c", width=2)
    ), 
    secondary_y=False
)

fig.update_layout(
    title=f"Smoothing Sandbox Visual Evaluation: SMA vs EMA vs Median (Days {start_day} to {end_day}, Window: {window_size})",
    template="plotly_white",
    hovermode="x unified",
    xaxis_title="Day Number",
    yaxis_title="Price"
)

fig.show()